# Consulta de profesionales de la salud — Servicio Practitioner (MSAL)

Cliente en Python para el microservicio **Practitioner** (HL7 FHIR) del
**Bus de Interoperabilidad del Ministerio de Salud de la Nación (Argentina)**.
Permite autenticarse contra el bus y buscar profesionales registrados en
**REFEPS** por DNI, CUIL, código REFEPS o ID interno.

### Flujo

```
┌──────────────┐  1. JWT firmado (HS256)   ┌────────────────────────┐
│              │ ────────────────────────▶ │ POST /bus-auth/v2/auth │
│   Cliente    │ ◀──────────────────────── │                        │
│  (notebook)  │     2. accessToken        └────────────────────────┘
│              │  3. GET + Bearer token    ┌────────────────────────┐
│              │ ────────────────────────▶ │ GET /fhir/Practitioner │
│              │ ◀──────────────────────── │                        │
└──────────────┘  4. Bundle / Practitioner └────────────────────────┘
```

### Requisitos

- Python 3.10+
- `requests` (`pip install requests`)
- Un **dominio habilitado** en el Bus de Interoperabilidad, con su
  `DOMAIN_URL` y `DOMAIN_TOKEN_SECRET` (los otorga el MSAL).

### Referencias

- *Guía sobre servicio Practitioner v2.0* (MSAL, septiembre 2025).
- Colección Postman `PractitionerPROD` provista por el MSAL.

In [ ]:
import base64
import hashlib
import hmac
import json
import os
import time
from typing import Any

import requests

## 1. Configuración

| Variable | Descripción | Default |
|---|---|---|
| `BUS_URL` | URL base del Bus de Interoperabilidad | `https://bus.msal.gob.ar` (producción) |
| `DOMAIN_URL` | Dominio registrado ante el MSAL (va como `iss` del JWT) | — |
| `DOMAIN_TOKEN_SECRET` | Secreto compartido para firmar el JWT | — |

Las credenciales se leen de un archivo **`.env`** en la carpeta de la
notebook (si existe) o de variables de entorno con el mismo nombre:

```bash
cp .env.example .env   # y completar los valores
```

```ini
PRACTITIONER_BUS_URL=https://bus.msal.gob.ar
PRACTITIONER_DOMAIN_URL=https://tu-dominio.com.ar
PRACTITIONER_DOMAIN_SECRET=...
```

`.env` está en el `.gitignore`: el secreto nunca se versiona.

In [ ]:
def load_env_file(path: str = ".env") -> None:
    """Carga `CLAVE=valor` de un archivo .env sin pisar variables ya definidas."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file()

BUS_URL = os.environ.get("PRACTITIONER_BUS_URL", "https://bus.msal.gob.ar")
DOMAIN_URL = os.environ.get("PRACTITIONER_DOMAIN_URL")
DOMAIN_TOKEN_SECRET = os.environ.get("PRACTITIONER_DOMAIN_SECRET")

REQUEST_TIMEOUT = 30  # segundos

if not DOMAIN_URL or not DOMAIN_TOKEN_SECRET:
    raise RuntimeError(
        "Faltan PRACTITIONER_DOMAIN_URL y/o PRACTITIONER_DOMAIN_SECRET. "
        "Copiá .env.example a .env y completalo."
    )
print(f"Dominio: {DOMAIN_URL} | Bus: {BUS_URL}")

## 2. Autenticación

### 2.1 `clientAssertion` (JWT)

Según la guía, el bus espera un JWT firmado con HS256 usando
`DOMAIN_TOKEN_SECRET`:

```json
// header
{ "alg": "HS256", "typ": "JWT" }

// payload
{
  "iss": "<DOMAIN_URL>",
  "iat": <timestamp>,
  "exp": <timestamp + N>,
  "aud": "aud", "sub": "sub", "name": "name", "ident": "ident", "role": "role"
}
```

> **`iat` / `exp` en segundos.** La guía dice que el timestamp va en
> milisegundos, pero tanto el JWT de ejemplo del PDF como el pre-request
> script de Postman (`Math.floor(Date.now() / 1000)`) usan segundos (Unix
> time estándar). Esta implementación usa segundos, que es lo que el bus
> acepta en la práctica.

La firma se arma con la librería estándar (`hmac` + `hashlib`), sin depender
de PyJWT.

In [ ]:
def _base64url(raw: bytes) -> str:
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode("ascii")


def build_client_assertion(domain_url: str, secret: str, ttl_seconds: int = 3600) -> str:
    """Arma y firma (HS256) el JWT que va en el campo `clientAssertion`."""
    now = int(time.time())
    header = {"alg": "HS256", "typ": "JWT"}
    payload = {
        "iss": domain_url,
        "iat": now,
        "exp": now + ttl_seconds,
        "aud": "aud",
        "sub": "sub",
        "name": "name",
        "ident": "ident",
        "role": "role",
    }

    def encode(part: dict) -> str:
        return _base64url(json.dumps(part, separators=(",", ":")).encode("utf-8"))

    signing_input = f"{encode(header)}.{encode(payload)}"
    signature = hmac.new(secret.encode("utf-8"), signing_input.encode("ascii"), hashlib.sha256).digest()
    return f"{signing_input}.{_base64url(signature)}"

### 2.2 Obtención del `accessToken`

`POST {BUS_URL}/bus-auth/v2/auth` con el JWT anterior. La guía dice que el
token viene en el atributo `token`, pero la colección Postman lee
`accessToken`, así que se aceptan los dos.

In [ ]:
def get_access_token(
    domain_url: str,
    secret: str,
    bus_url: str = BUS_URL,
    session: requests.Session | None = None,
) -> str:
    """Intercambia el `clientAssertion` por un `accessToken` del bus."""
    http = session or requests
    body = {
        "grantType": "client_credentials",
        "scope": "Practitioner/*.read",
        "clientAssertionType": "urn:ietf:params:oauth:client-assertion-type:jwt-bearer",
        "clientAssertion": build_client_assertion(domain_url, secret),
    }
    resp = http.post(f"{bus_url}/bus-auth/v2/auth", json=body, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()

    data = resp.json()
    token = data.get("accessToken") or data.get("token")
    if not token:
        raise KeyError(f"La respuesta de /bus-auth/v2/auth no trae 'accessToken' ni 'token': {data}")
    return token

## 3. Cliente del servicio Practitioner

Endpoints documentados en la guía (todos `GET`, con `Authorization: Bearer <token>`):

| Búsqueda | Request | Respuesta |
|---|---|---|
| Código REFEPS | `/fhir/Practitioner?identifier=https://sisa.msal.gov.ar/REFEPS\|{codigo}` | `Bundle` |
| DNI + género | `/fhir/Practitioner?identifier=https://www.renaper.org.ar/dni\|{dni}&gender={male\|female}` | `Bundle` |
| CUIL (sin guiones) | `/fhir/Practitioner?identifier=https://www.anses.org.ar/cuil\|{cuil}` | `Bundle` |
| ID interno | `/fhir/Practitioner/{id}` | `Practitioner` |

`PractitionerClient` pide el token la primera vez que se lo necesita y
reutiliza la misma conexión HTTP (`requests.Session`) en todas las consultas.

In [ ]:
SYSTEM_REFEPS = "https://sisa.msal.gov.ar/REFEPS"
SYSTEM_DNI = "https://www.renaper.org.ar/dni"
SYSTEM_CUIL = "https://www.anses.org.ar/cuil"


class PractitionerClient:
    """Cliente del servicio Practitioner del Bus de Interoperabilidad (MSAL)."""

    def __init__(self, domain_url: str, secret: str, bus_url: str = BUS_URL):
        self.domain_url = domain_url
        self.bus_url = bus_url.rstrip("/")
        self._secret = secret
        self._token: str | None = None
        self.session = requests.Session()

    def __repr__(self) -> str:
        return f"PractitionerClient(domain_url={self.domain_url!r}, bus_url={self.bus_url!r})"

    # --- autenticación -------------------------------------------------
    def authenticate(self) -> None:
        self._token = get_access_token(self.domain_url, self._secret, self.bus_url, self.session)

    def _get(self, path: str, params: dict | None = None) -> requests.Response:
        if self._token is None:
            self.authenticate()
        return self.session.get(
            f"{self.bus_url}{path}",
            headers={"Authorization": f"Bearer {self._token}"},
            params=params,
            timeout=REQUEST_TIMEOUT,
        )

    def _search(self, system: str, value: str, **extra: str) -> requests.Response:
        return self._get("/fhir/Practitioner", params={"identifier": f"{system}|{value}", **extra})

    # --- búsquedas -----------------------------------------------------
    def buscar_por_refeps(self, codigo_refeps: str) -> requests.Response:
        return self._search(SYSTEM_REFEPS, codigo_refeps)

    def buscar_por_cuil(self, cuil: str) -> requests.Response:
        return self._search(SYSTEM_CUIL, cuil.replace("-", ""))

    def buscar_por_id(self, practitioner_id: str) -> requests.Response:
        return self._get(f"/fhir/Practitioner/{practitioner_id}")

    def buscar_por_dni(self, dni: str, gender: str | None = None) -> requests.Response:
        """
        Busca por DNI. La guía documenta el género (`male` / `female`) como
        obligatorio; si no se pasa, se prueba sin género y después con cada
        valor, y se devuelve la primera respuesta con resultados.
        """
        intentos = [gender] if gender else [None, "male", "female"]
        resp = None
        for g in intentos:
            extra = {"gender": g} if g else {}
            resp = self._search(SYSTEM_DNI, dni, **extra)
            if resp.ok and _tiene_resultados(resp):
                return resp
        return resp


def _tiene_resultados(resp: requests.Response) -> bool:
    try:
        data = resp.json()
    except ValueError:
        return False
    return data.get("resourceType") == "Practitioner" or data.get("total", 0) > 0

## 4. Lectura de la respuesta

Las búsquedas por `identifier` devuelven un `Bundle` y la búsqueda por ID
devuelve un `Practitioner` directo. `extraer_practitioners` lleva los dos
casos a una lista de recursos, y `resumir_practitioner` saca de cada uno los
datos útiles, entre ellos las matrículas (`qualification`) con sus
extensiones `MatriculaHabilitada`, `jurisdMatricula` y `especialidadMatricula`.

In [ ]:
def _extension_value(extensions: list | None, url_suffix: str) -> Any:
    """Devuelve el `value[x]` de la primera extensión cuya URL termina en `url_suffix`."""
    for ext in extensions or []:
        if ext.get("url", "").endswith(url_suffix):
            for key, val in ext.items():
                if key.startswith("value"):
                    return val
    return None


def _display(value: Any) -> Any:
    """Saca `display` de un Coding / CodeableConcept, o devuelve el valor tal cual."""
    if isinstance(value, dict):
        if "display" in value:
            return value["display"]
        if value.get("coding"):
            return value["coding"][0].get("display")
    return value


def extraer_practitioners(data: dict) -> list[dict]:
    if data.get("resourceType") == "Practitioner":
        return [data]
    if data.get("resourceType") == "Bundle":
        return [e["resource"] for e in data.get("entry", []) if "resource" in e]
    return []


def resumir_practitioner(resource: dict) -> dict:
    nombre = (resource.get("name") or [{}])[0]
    nombre_completo = nombre.get("text") or " ".join([*nombre.get("given", []), nombre.get("family", "")]).strip()

    matriculas = []
    for q in resource.get("qualification", []):
        exts = q.get("extension")
        matriculas.append({
            "profesion": _display(q.get("code")),
            "matricula": (q.get("identifier") or [{}])[0].get("value"),
            "jurisdiccion": _display(_extension_value(exts, "jurisdMatricula")),
            "habilitada": _extension_value(exts, "MatriculaHabilitada"),
            "especialidades": [
                _display(e.get("valueCoding") or e.get("valueCodeableConcept"))
                for e in exts or []
                if e.get("url", "").endswith("especialidadMatricula")
            ],
        })

    return {
        "id": resource.get("id"),
        "nombre": nombre_completo,
        "genero": resource.get("gender"),
        "fecha_nacimiento": resource.get("birthDate"),
        "activo": resource.get("active"),
        "identificadores": {i.get("system"): i.get("value") for i in resource.get("identifier", [])},
        "matriculas": matriculas,
    }


def mostrar_respuesta(resp: requests.Response) -> None:
    """Imprime un resumen legible de la respuesta del servicio."""
    print(f"HTTP {resp.status_code}")
    if not resp.ok:
        print(resp.text)
        return

    practitioners = extraer_practitioners(resp.json())
    print(f"Resultados: {len(practitioners)}")
    for p in map(resumir_practitioner, practitioners):
        print(f"\n{p['nombre']}  (id {p['id']})")
        print(f"  Género: {p['genero']} | Nacimiento: {p['fecha_nacimiento']} | Activo: {p['activo']}")
        print("  Identificadores:")
        for system, value in p["identificadores"].items():
            print(f"    - {system}: {value}")
        print("  Matrículas:")
        for m in p["matriculas"]:
            print(f"    - {m['profesion'] or '(sin profesión)'} | matrícula {m['matricula']} | "
                  f"jurisdicción: {m['jurisdiccion'] or '-'} | habilitada: {m['habilitada']}")
            if m["especialidades"]:
                print(f"      especialidades: {', '.join(filter(None, m['especialidades']))}")

## 5. Ejemplo de uso

In [ ]:
client = PractitionerClient(DOMAIN_URL, DOMAIN_TOKEN_SECRET)
client.authenticate()
client

In [ ]:
DNI = "12345678"  # reemplazar por el DNI a consultar

resp = client.buscar_por_dni(DNI)
mostrar_respuesta(resp)

Otras búsquedas disponibles:

```python
client.buscar_por_cuil("20-12345678-9")
client.buscar_por_refeps("541012345678")
client.buscar_por_id("1234567")
client.buscar_por_dni("12345678", gender="female")
```

### Respuesta FHIR completa

El JSON tal como lo devuelve el servicio, útil para depurar o para ver campos
que el resumen no muestra.

In [ ]:
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))